# Module 2 — Microservices: guided beginner activities

This notebook builds on **Module 1: REST APIs**. You will use the same request-and-response ideas with two cooperating services: one for accounts and authentication, and one for URL shortening.

Use this notebook as the primary step-by-step activity. `upstream-participants-guide.md` remains available as a reference for workshop setup and service information.

The activities follow a cycle: **predict → run → observe → explain → reflect**. Pause at each prompt before running the next code cell.

## Workshop learning outcomes

By the end, you should be able to:

- explain how responsibilities are divided between the authentication service and the shortener service;
- register an account, log in, and use the returned token in an HTTP request;
- explain what a signed JWT does—and why it is **not encrypted**;
- distinguish authentication (who is making a request?) from authorization (what may they do?);
- test the difference between protected management operations and a public redirect; and
- identify limitations in this educational implementation.

### Before you start
Use workshop-only usernames and passwords. Do not reuse a personal password. The notebook keeps credentials and tokens in variables and avoids printing them, but do not share or commit notebook outputs containing sensitive data. Never copy a private key into a notebook or the shortener service.

## 1. What carries over from Module 1?

**Predict.** Think of an operation you used in Module 1. Without running anything, note its HTTP method, path, request data (if any), and the kind of response you expected.

1. Which of those HTTP ideas should still apply when an API is split across services?
2. What might become more complicated when a request involves more than one service?

**Checkpoint.** HTTP methods, paths, status codes, headers, and JSON request/response bodies still matter. Module 2 adds another question: which service is responsible for each part of the behavior?

## 2. Predict the service responsibilities

There are two participant-local services:

| Service | Address | First prediction: what should it own? |
|---|---|---|
| Authentication | `http://127.0.0.1:5001` | Accounts, passwords, and issuing tokens |
| Shortener | `http://127.0.0.1:5000` | URL mappings, ownership checks, and redirects |

**Predict.** If a user creates a shortened URL, which service should check the user's identity? Which service should store the mapping? Write down your prediction before continuing.

**Explain.** The authentication service signs a token using its private key. For protected management requests, the shortener verifies that signature using a copy of the public key. It does not contact the authentication service for a live account lookup on every request.

## 3. Start the supplied services

The service implementations are already provided. In this workshop you will send requests to them and inspect their code; you do not need to rewrite them.

From the Module 2 beginner directory, open a terminal and prepare the local keys:

```bash
bash setup_local_keys.sh
```

Then start the authentication service in one terminal:

```bash
cd auth_service
python auth.py
```

Start the shortener service in a second terminal:

```bash
cd shortener_service
python shortener.py
```

Keep both terminals running while you work. The services keep users and mappings in memory, so restarting a service clears its own data. The RSA keys are stored separately on disk by the setup script.

**Predict.** The next cell sends requests that should be rejected. Why can a rejected request still show that a service is running?

In [ ]:
import base64
import json
import secrets
import getpass
import requests

AUTH_URL = "http://127.0.0.1:5001"
SHORTENER_URL = "http://127.0.0.1:5000"

print("Notebook helpers are ready. No credentials or tokens are displayed.")

In [ ]:
# An empty login request should be rejected by the running authentication service.
auth_probe = requests.post(f"{AUTH_URL}/users/login", json={})

# Listing mappings without a token should be rejected by the running shortener.
shortener_probe = requests.get(f"{SHORTENER_URL}/")

print("Authentication service response status (expected 400):", auth_probe.status_code)
print("Shortener response status (expected 403):", shortener_probe.status_code)

**Observe and explain.** A `400` or `403` here is expected: the services received the requests and applied their rules. It is different from a connection error, which would mean the service could not be reached. A working network connection does not itself mean a request is authorized.

## 4. Register an account

The authentication service accepts `POST /users` with a JSON body containing `username` and `password`.

**Predict.** What status code do you expect when a new username is registered successfully? What might happen if the same username is registered again?

The next cell creates a fresh workshop username so it can be rerun without colliding with an earlier notebook run. Enter a workshop-only password when prompted; it will not be echoed.

In [ ]:
run_id = secrets.token_hex(4)
username_a = f"learner-a-{run_id}"
password_a = getpass.getpass("Enter a workshop-only password for learner A: ")

registration_a = requests.post(
    f"{AUTH_URL}/users",
    json={"username": username_a, "password": password_a},
)
print("Status:", registration_a.status_code)
print("Response:", registration_a.json())

**Observe.** A successful registration returns `201`. The service stores a password hash in its in-memory user dictionary, not the plain-text password.

Open `auth_service/auth.py` in the Jupyter file browser and find `create_user`. Match the JSON fields and status code you just observed to the route implementation.

**Explain.** What information does the authentication service now have? Has the shortener service been asked to create a URL mapping yet?

### Try the duplicate-registration case

**Predict.** The next request repeats the same registration. Should it create another account, replace the password, or reject the duplicate?

In [ ]:
duplicate_registration = requests.post(
    f"{AUTH_URL}/users",
    json={"username": username_a, "password": password_a},
)
print("Status (expected 409):", duplicate_registration.status_code)
print("Response:", duplicate_registration.json())

**Observe and reflect.** The service returns `409` for a duplicate username. Why is it useful for an API to make this case distinguishable from a successful creation? Find the corresponding check in `create_user`.

## 5. Log in and inspect the token

Login also uses `POST /users`, specifically `POST /users/login`, with `username` and `password`. A successful response contains a token.

**Predict.** The next cell will not print the token. What should the client keep so it can use the token in a later request? What should the service return if the password is wrong?

In [ ]:
login_a = requests.post(
    f"{AUTH_URL}/users/login",
    json={"username": username_a, "password": password_a},
)
print("Status:", login_a.status_code)

token_a = login_a.json().get("token") if login_a.status_code == 200 else None
print("Token stored in token_a:", token_a is not None)
print("Token value intentionally not displayed.")

### What can you learn from a JWT?

A JWT has three dot-separated parts. Its header and payload are Base64url-encoded, which is an encoding—not encryption. The next cell decodes the first two parts locally. It does **not** verify the signature.

**Predict.** Which identity and time-related claims do you expect to see?

In [ ]:
def decode_jwt_part(part):
    padded = part + "=" * (-len(part) % 4)
    return json.loads(base64.urlsafe_b64decode(padded.encode("utf-8")))

if token_a is None:
    print("Login did not succeed; rerun the login cell after checking the service.")
else:
    token_parts = token_a.split(".")
    jwt_header = decode_jwt_part(token_parts[0])
    jwt_payload = decode_jwt_part(token_parts[1])
    print("Header:", jwt_header)
    print("Payload claim names:", sorted(jwt_payload.keys()))
    print("Subject matches this workshop account:", jwt_payload.get("sub") == username_a)
    print("Issued-at value (Unix time):", jwt_payload.get("iat"))
    print("Expiry value (Unix time):", jwt_payload.get("exp"))

**Observe and explain.** The token includes `sub`, `iat`, and `exp`. The header identifies `RS256`; the authentication service signs the encoded header and payload with its RSA private key. The shortener can verify the signature using the public key.

A signature helps detect changes to signed content; it does not hide that content. Do not put secrets in a JWT payload. Decoding the payload in the previous cell was only inspection, not proof that the token is valid.

Find `generate_jwt` in `auth_service/auth.py`. Which key signs the token? Why should the private key stay in the authentication service?

## 6. Compare a protected request with and without a token

The shortener's `GET /` lists mapping IDs for the authenticated user.

**Predict.** What response should the shortener return when this request has no token? What changes when it has learner A's token?

In [ ]:
anonymous_list = requests.get(f"{SHORTENER_URL}/")
print("Without a token (expected 403):", anonymous_list.status_code, anonymous_list.json())

headers_a = {"Authorization": f"Bearer {token_a}"}
authenticated_list = requests.get(f"{SHORTENER_URL}/", headers=headers_a)
print("With learner A's token:", authenticated_list.status_code, authenticated_list.json())

**Observe.** The token is sent in the `Authorization` header using the `Bearer` scheme. A missing or invalid token is rejected with `403`; a valid token can access the protected listing.

Open `shortener_service/shortener.py` and find `get_token_from_request`, `verify_jwt`, `get_current_user`, and `read_root`. Trace how the request header becomes a username and then a list of that user's IDs. The shortener checks the token locally; it does not ask the authentication service whether the account still exists.

## 7. Create a mapping and inspect a public redirect

The shortener accepts `POST /` with a JSON field called `url`. A successful response contains the generated short ID.

**Predict.** What should happen if you send this request with learner A's token? Which status code signals a new resource was created?

In [ ]:
long_url_a = "https://example.org/module2/learner-a"
create_a = requests.post(
    f"{SHORTENER_URL}/",
    headers=headers_a,
    json={"url": long_url_a},
)
print("Status (expected 201):", create_a.status_code)
print("Response:", create_a.json())
short_id_a = create_a.json().get("id") if create_a.status_code == 201 else None

**Predict.** A shortened URL should send a browser to the original URL. What HTTP status and response header would let a client know where to go? The next request disables automatic redirect-following so you can inspect the response itself.

In [ ]:
if short_id_a is not None:
    redirect_response = requests.get(
        f"{SHORTENER_URL}/{short_id_a}",
        allow_redirects=False,
    )
    print("Status (expected 301):", redirect_response.status_code)
    print("Location header:", redirect_response.headers.get("Location"))
else:
    print("Create the mapping successfully before testing its redirect.")

**Observe and explain.** `GET /<id>` is public: it does not require a token or check mapping ownership. For an existing ID, this implementation returns a real `301` redirect with a `Location` header.

Compare this with Module 1: its redirect-like response used status `301` but returned JSON without a `Location` header. Why does the header matter to a browser or HTTP client? Find `read_item` in `shortener_service/shortener.py` and connect its `redirect(...)` call to the response you observed.

## 8. Authentication is not the same as authorization

Authentication asks, “Who is making this request?” Authorization asks, “Is that identity allowed to perform this operation on this resource?”

**Predict.** Learner B will have a valid token, but learner A owns the mapping. Should B be able to list A's IDs or update A's mapping?

In [ ]:
username_b = f"learner-b-{secrets.token_hex(4)}"
password_b = getpass.getpass("Enter a different workshop-only password for learner B: ")

registration_b = requests.post(
    f"{AUTH_URL}/users",
    json={"username": username_b, "password": password_b},
)
login_b = requests.post(
    f"{AUTH_URL}/users/login",
    json={"username": username_b, "password": password_b},
)
print("Registration B status (expected 201):", registration_b.status_code)
print("Login B status (expected 200):", login_b.status_code)
token_b = login_b.json().get("token") if login_b.status_code == 200 else None
headers_b = {"Authorization": f"Bearer {token_b}"} if token_b else {}
print("Learner B token stored without displaying it:", token_b is not None)

In [ ]:
list_a = requests.get(f"{SHORTENER_URL}/", headers=headers_a)
list_b = requests.get(f"{SHORTENER_URL}/", headers=headers_b)
print("Learner A's list:", list_a.status_code, list_a.json())
print("Learner B's list:", list_b.status_code, list_b.json())

if short_id_a is not None and token_b is not None:
    cross_user_update = requests.put(
        f"{SHORTENER_URL}/{short_id_a}",
        headers=headers_b,
        json={"url": "https://example.org/not-owned-by-b"},
    )
    print("Learner B updating A's mapping (expected 403):", cross_user_update.status_code)

**Observe and explain.** Both learners can be authenticated, but their mapping lists are separate. B's cross-user update should return `403`: having a valid token does not grant ownership of A's mapping.

Now try the public redirect to A's ID without a token. It should still work while that mapping exists. What does this tell you about the difference between a public operation and a protected management operation?

In the source, find the owner check in `update_item` and the absence of an ownership check in `read_item`. Explain why each route makes a different choice.

In [ ]:
if short_id_a is not None:
    public_as_b = requests.get(f"{SHORTENER_URL}/{short_id_a}", allow_redirects=False)
    print("Public redirect without a token (expected 301):", public_as_b.status_code)

missing_public = requests.get(f"{SHORTENER_URL}/no-such-id", allow_redirects=False)
print("Unknown public ID (expected 404):", missing_public.status_code, missing_public.json())

## 9. What happens when a token is missing or changed?

**Predict.** The next cell changes one character in the signature portion of learner A's token. The payload is not changed. Will the shortener accept the request? Why?

The full token stays in memory and is not printed. This is a local teaching test; do not paste real tokens into shared logs or documents.

In [ ]:
if token_a is not None:
    parts = token_a.split(".")
    changed_first_character = "A" if parts[2][0] != "A" else "B"
    tampered_token = ".".join([parts[0], parts[1], changed_first_character + parts[2][1:]])
    tampered_headers = {"Authorization": f"Bearer {tampered_token}"}
    tampered_response = requests.get(f"{SHORTENER_URL}/", headers=tampered_headers)
    print("Modified-token response (expected 403):", tampered_response.status_code, tampered_response.json())
else:
    print("Log in successfully before trying the modified-token experiment.")

**Observe and explain.** The modified token should be rejected because its signature no longer matches the signed content. This demonstrates integrity checking; it does not make the payload secret.

The shortener checks the token signature and expiry and requires a subject. This teaching verifier does not validate the JWT header, issuer, audience, or subject type, and it does not check whether the user still exists. It also has no token-revocation mechanism. These are implementation limitations, not recommendations for a production verifier.

## 10. Connect the behavior to the implementation

Use the Jupyter file browser to open the two existing service files. Do not edit them for this activity. Find each function and connect one observation from the notebook to the relevant code:

| Behavior you observed | Function to find |
|---|---|
| A new account is created | `create_user` in `auth_service/auth.py` |
| A token is constructed and signed | `generate_jwt` in `auth_service/auth.py` |
| The token header is read and verified | `get_token_from_request` and `verify_jwt` in `shortener_service/shortener.py` |
| A user's IDs are listed | `read_root` in `shortener_service/shortener.py` |
| An owned mapping can be updated | `update_item` in `shortener_service/shortener.py` |
| An ID redirects publicly | `read_item` in `shortener_service/shortener.py` |

### Security and design review

Choose one observed implementation choice. Complete the three steps in your own notes:

1. **Observed behavior:** What does the code do?
2. **Risk or trade-off:** What limitation or consequence follows?
3. **Possible improvement:** What safer or more robust approach could be used?

Confirmed limitations you can investigate include unsalted SHA-256 password hashing, Flask debug mode, in-memory users and mappings, no token revocation, incomplete JWT claim/header validation, and URL validation that checks only for an `http://` or `https://` prefix. This is educational code, not a production-ready authentication service or web deployment.

## 11. Run the supplied tests and review the contract

The supplied tests exercise selected service behaviors, including authentication and ownership isolation. They are useful evidence about the behaviors they test, but passing tests is not proof that a system is secure.

From a terminal whose current directory is `Module-2/beginner`, run the test suite with both services still running:

```bash
cd ../shared/tests
python test_microservices.py
```

Before running it, predict which behaviors its test names cover. Afterward, compare your prediction with the output and test file. The test run may create accounts or mappings; user accounts remain in the authentication service's memory until that service restarts.

One unusual assignment contract is that successful `DELETE /`—deleting all mappings owned by the caller—returns status `404`. The single-mapping delete route returns `204` on success. Discuss why an API response can be part of an assignment contract while still being unusual REST semantics. Do not treat the bulk-delete status as a general design recommendation.

## 12. Supported practice and review

Answer in your own words:

1. Which responsibilities belong to the authentication service, and which belong to the shortener?
2. What does the JWT signature let the shortener check? What does it **not** hide?
3. How did you observe the difference between authentication and authorization?
4. Why did a public redirect work without an owner token, while updating another user's mapping did not?
5. How did the Module 2 redirect response differ from the redirect-like response in Module 1?
6. Name one limitation you found in the code and propose one improvement.
7. What new coordination or failure modes appear when one API is split into multiple services?

### Optional challenge

Inspect `PUT /users` in `auth_service/auth.py`. Predict what happens to an already-issued token after the password changes. Then trace the shortener's verifier to see whether it contacts the authentication service or maintains a revocation list. Explain your reasoning without sharing the token or password.